# Planning for Annual Expenses

Many financial obligations occur just once per year: property taxes, insurance premiums, annual subscriptions, and membership fees. This notebook shows how to plan for these annual expenses using the Sinking Fund system, optimizing contributions throughout the year to have funds ready when needed.

**Topics covered:**
- Property taxes, insurance premiums, annual subscriptions
- Multiple bills with different due dates
- Optimizing contributions across the year


In [ ]:
from datetime import date
from sinkingfund import SinkingFund


## Setting Up Annual Bills

Let's create a sinking fund for common annual expenses: property taxes, car insurance, and annual subscriptions.


In [ ]:
# Define annual expenses with their due dates.
annual_bills = [
    {
        'bill_id': 'property_tax',
        'service': 'Property Tax 2025',
        'amount_due': 3600.00,
        'recurring': False,
        'due_date': date(2025, 11, 1)
    },
    {
        'bill_id': 'car_insurance',
        'service': 'Annual Car Insurance Premium',
        'amount_due': 1200.00,
        'recurring': False,
        'due_date': date(2025, 8, 15)
    },
    {
        'bill_id': 'annual_subscription',
        'service': 'Software Subscription Annual',
        'amount_due': 599.00,
        'recurring': False,
        'due_date': date(2025, 6, 1)
    },
    {
        'bill_id': 'home_insurance',
        'service': 'Homeowners Insurance Annual',
        'amount_due': 1800.00,
        'recurring': False,
        'due_date': date(2025, 4, 1)
    }
]

# Create sinking fund for 2025.
fund = SinkingFund(
    start_date=date(2025, 1, 1),
    end_date=date(2025, 12, 31),
    balance=500.00  # Starting balance
)

# Load the bills.
fund.add_bills(annual_bills)

print("Annual expenses loaded:")
for bill in fund.get_bills():
    print(f"  {bill.service}: ${bill.amount_due:.2f} due {bill.start_date}")


## Generating the Plan

We'll use `quick_report()` to automatically create envelopes, allocate funds, schedule contributions, and generate a cash flow report. This gives us a complete plan optimized for bi-weekly contributions.


In [ ]:
# Generate complete plan with bi-weekly contributions.
report = fund.quick_report(contribution_interval=14)

print("=== Annual Expense Planning Report ===\n")
print(f"Planning period: {fund.start_date} to {fund.end_date}")
print(f"Starting balance: ${fund.balance}")
print(f"Total annual expenses: ${sum(bill.amount_due for bill in fund.get_bills()):.2f}\n")

# Show envelopes created.
print("Envelopes created:")
for envelope in fund.get_envelopes():
    print(f"  {envelope.bill_instance.service}")
    print(f"    Due: {envelope.bill_instance.due_date}")
    print(f"    Amount: ${envelope.bill_instance.amount_due:.2f}")
    print(f"    Contribution window: {envelope.start_contrib_date} to {envelope.end_contrib_date}")


## Analyzing Cash Flow

Let's examine the contribution schedule to see how funds accumulate throughout the year.


In [ ]:
# Analyze contribution patterns.
from collections import defaultdict

# Group contributions by month.
monthly_contribs = defaultdict(float)
monthly_payouts = defaultdict(float)

for report_date, data in report.items():
    month_key = report_date.strftime('%Y-%m')
    monthly_contribs[month_key] += float(data['contributions']['total'])
    monthly_payouts[month_key] += abs(float(data['payouts']['total']))

print("=== Monthly Cash Flow Summary ===\n")
print("Month       | Contributions | Payouts    | Net Flow")
print("-" * 55)

for month in sorted(monthly_contribs.keys()):
    contribs = monthly_contribs[month]
    payouts = monthly_payouts[month]
    net = contribs - payouts
    print(f"{month} | ${contribs:>12.2f} | ${payouts:>9.2f} | ${net:>9.2f}")


## Checking Funding Status

Let's verify that each annual expense will be fully funded by its due date.


In [ ]:
# Check funding status for each envelope.
print("=== Funding Status by Due Date ===\n")

for envelope in sorted(fund.get_envelopes(), key=lambda e: e.bill_instance.due_date):
    due_date = envelope.bill_instance.due_date
    target = envelope.bill_instance.amount_due
    
    # Get balance on due date.
    balance_on_due = report[due_date]['account_balance']['total']
    
    is_funded = balance_on_due >= target
    status = "✓ Fully Funded" if is_funded else "✗ Underfunded"
    
    print(f"{envelope.bill_instance.service} ({due_date})")
    print(f"  Target: ${target:.2f}")
    print(f"  Balance on due date: ${balance_on_due:.2f}")
    print(f"  Status: {status}\n")


## Optimizing Contribution Intervals

Different contribution intervals (weekly, bi-weekly, monthly) affect how smoothly funds accumulate. Let's compare approaches.


In [ ]:
# Compare different contribution intervals.
intervals = {
    'Weekly': 7,
    'Bi-weekly': 14,
    'Monthly': 30
}

print("=== Comparison of Contribution Intervals ===\n")

for interval_name, days in intervals.items():
    test_fund = SinkingFund(
        start_date=date(2025, 1, 1),
        end_date=date(2025, 12, 31),
        balance=500.00
    )
    test_fund.add_bills(annual_bills)
    test_report = test_fund.quick_report(contribution_interval=days)
    
    # Count total contributions.
    total_contribs = sum(float(data['contributions']['total']) for data in test_report.values())
    contribution_count = sum(1 for data in test_report.values() if data['contributions']['total'] > 0)
    
    print(f"{interval_name} ({days} days):")
    print(f"  Total contributions: ${total_contribs:.2f}")
    print(f"  Number of contribution dates: {contribution_count}")
    
    # Check if all bills are funded.
    all_funded = True
    for envelope in test_fund.get_envelopes():
        due_date = envelope.bill_instance.due_date
        balance = test_report[due_date]['account_balance']['total']
        if balance < envelope.bill_instance.amount_due:
            all_funded = False
            break
    
    print(f"  All bills funded: {'Yes' if all_funded else 'No'}\n")


## Summary

**Key Strategies for Annual Expense Planning:**

1. **Early Planning**: Start saving well in advance of due dates to spread contributions evenly
2. **Bi-weekly Contributions**: Often align with paychecks and provide smooth accumulation
3. **Priority Allocation**: Funds are automatically allocated to earliest due dates first
4. **Automated Scheduling**: The system creates optimized contribution schedules automatically

**Benefits:**

- **No Surprises**: Funds are ready when bills come due
- **Smooth Cash Flow**: Regular contributions prevent large one-time payments
- **Peace of Mind**: Clear visibility into funding status throughout the year

**Next Steps:**
- Adjust contribution intervals to match your income schedule
- Add or modify bills as your annual obligations change
- Use allocation strategies to prioritize critical expenses
